<a href="https://colab.research.google.com/github/maitrungtung/IMIS_Machine_Learning/blob/main/Report/IMISToolA2026_Report5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Report 5 (2026/08/03 ver.A)

for Tools for intelligent interaction systems a (0ALE005 / 0AL5707).

---

* Student ID: 202520857
* Name: Tung Mai Trung
* Colab account: maitrungtung2602@gmail.com

---

Report could be written in English or Japanese. / レポートの記述は日本語でも英語でもよい．

---

**Note that this Report 5 is in conjunction with the IMIS Tool Exercise-A Lesson 700.  
Answers not linked with the Lesson 700 will be degraded.**  
https://github.com/kameda-yoshinari/IMISToolExeA/tree/main/700

---
# Report5A: Make your own classifier (Mandatory:必須)  

**Directly change "--" in this text cell to write a report.**

You are strongly encouraged to take images at your USB camera (or your smartphone camera).
(It is because the dataset would be obvious and trivial if you collect images by the google query with the word...)   
(DO NOT USE open dataset!)

* Show the link to the top google-drive folder of the image dataset (It should include train/Class1 train/Class2 val/Class1 val/Class2)

1. URL: -- (it should be accessible to kameda.yoshinari@image.iit.tsukuba.ac.jp / google account)

* Show the two class names you give.

1. Class1: --
2. Class2: --

* Number of images

1. Class1: --
2. Class2: --

* Three examples of test results (true label, two predicted labels of model_ft and model_conv, image)

1. Example1: --
2. Example2: --
3. Example3: --

Images should be given by URL that is accesible by kameda.yoshinari.ft@u.tsukuba.ac.jp, or inline image).


Write your findings of Report5A (Any comments are OK) below.

--

## **Report 5A — My Own Classifier**

This experiment follows the transfer-learning procedure introduced in **IMIS Tool Exercise-A Lesson 700**.

Lesson 700 introduces two approaches using a pretrained ResNet18:

- `model_ft`: fine-tuning all layers of the pretrained network.
- `model_conv`: using the pretrained network as a fixed feature extractor and training only the final classification layer.

I applied the same procedure to my own two-class image dataset. All images were taken by myself using a camera.

### Dataset

URL: **https://drive.google.com/drive/folders/1uDdgqlvsI8DscCtITQ8RIk_yfAwIx_jM?usp=drive_link**

Class1: **bottle**  
Class2: **calculator**

Number of images:

- bottle: 20 images = 15 training + 5 validation
- calculator: 20 images = 15 training + 5 validation

Dataset structure:

    train/
        bottle/
        calculator/
    val/
        bottle/
        calculator/

## **Preparation as Google Colab**


All the files will be placed on your Google Drive.

In [5]:
# Report 5A - Setup
from google.colab import drive
drive.mount('/content/drive')

import os, time, copy
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torchvision import datasets, models, transforms
import torchvision

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch: 2.11.0+cpu
Device: cpu


### **Loading and Preprocessing My Dataset**

Following Lesson 700, I use data augmentation for the training images and ImageNet normalization for both training and validation images.

Training images are randomly cropped and horizontally flipped, while validation images use deterministic resizing and center cropping. Both are converted into 224 × 224 RGB tensors suitable for the pretrained ResNet18 model.

In [6]:
# Report 5A - Load my bottle/calculator dataset

data_dir = "/content/drive/My Drive/IMIS_Tool-A/Work700/data/hymenoptera_data"

data_transforms = {
    "train": transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )
    ]),

    "val": transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )
    ])
}

image_datasets = {
    x: datasets.ImageFolder(
        os.path.join(data_dir, x),
        data_transforms[x]
    )
    for x in ["train", "val"]
}

dataloaders = {
    x: torch.utils.data.DataLoader(
        image_datasets[x],
        batch_size=4,
        shuffle=True,
        num_workers=2
    )
    for x in ["train", "val"]
}

dataset_sizes = {
    x: len(image_datasets[x])
    for x in ["train", "val"]
}

class_names = image_datasets["train"].classes

print("Dataset sizes:", dataset_sizes)
print("Class names:", class_names)
print("Device:", device)

Dataset sizes: {'train': 30, 'val': 10}
Class names: ['bottle', 'calculator']
Device: cpu


In [ ]:
# Visualize some training images

inputs, classes = next(iter(dataloaders["train"]))
grid = torchvision.utils.make_grid(inputs)

img = grid.numpy().transpose((1, 2, 0))
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
img = std * img + mean
img = np.clip(img, 0, 1)

plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.title(" | ".join(class_names[x] for x in classes))
plt.axis("off")
plt.show()

---
# Report5X: Evaluate the performance by "test" dataset (Optional:発展)  

Prepare 10 test images that take either of the two class objects (or other if you prefer, in this case, the 3rd label should be "other") and measure the performance of the two classifiers (of model_ft and model_conv).

* The images should be placed at the same folder as shown in 5A. (e.g. test/Class1 test/Class2 test/other)
* The total number of images in test/ folder should be 10 or more.
* Show the accuracy for the 10 images for both model_ft and model_conv.
* Carefully write the table and the report so as to be easy to read.


Report of 5X should be placed by changing this text cell (you can add more cells if you need).

---
# Report5Y: Evaluate the performance by changing the training (and validation) dataset (Optional:発展)  

By changinge the number of the images in train / validation, show the classification ratio for each.

* The number changing policy is up to the student.
* Write the way to select the images.
* Discuss what you expect before you start and what actually happend in the experimental result.
* Guess the reason if you encounter the expectation does not become true.


Report of 5Y should be placed by changing this text cell (you can add more cells if you need).

---
# Report5Z: Hide the objects and see what happens on the classifiers (Optional:発展)  

You may think the classifier could be build even when the objects are deleted from the training images as the some peripheral areas might have the biased image property correlated to the object.

* Delete (black out) the objects roughly in the training images.
* Run the same process of 5A, and 5X / 5Y if possible.
* Discuss what you expect before you start and what actually happend in the experimental result.
* Justify the reason of what actually happend.

Further reading:


[Object Recognition with and without Objects (2016)](https://arxiv.org/abs/1611.06596)  
[Noise or Signal: The Role of Image Backgrounds in Object Recognition (2017)](https://arxiv.org/abs/2006.09994)  
  

A discussion before AI/DL age  
[物体検出 — 背景と検出対象のモデリング — (2005)](https://vision.kuee.kyoto-u.ac.jp/japanese/happyou/pdf/Sumi_2005_P_197.pdf)

Report of 5Z should be placed by changing this text cell (you can add more cells if you need).

---
# Report submission

The report template will be given in ipynb file.  

You should save this file as a report templete to your local google colaboratory folder and then edit it to fit your report.

The report submission should be made at this cource (0ALE005) at https://manaba.tsukuba.ac.jp .  
Note that 0AL5707 is coupled with 0ALE005 on manaba system, so 0AL5707 students should also submit the report at 0ALE005.  
File extension should be **ipynb**. Other format won't be accepted.  








---
Tools and Practices for Intelligent Interaction Systems A  
Master's and Docotal programs in intelligent and mechanical interaction systems, University of Tsukuba, Japan.  
KAMEDA Yoshinari, SHIBUYA Takeshi  

知能システムツール演習a  
知能機能システム学位プログラム (筑波大学大学院)  
担当：亀田能成，澁谷長史  

2026/08/03. Ver.A.  


